In [ ]:
import certifi
import io as io
import re
import subprocess
import tempfile
import os

import pandas as pd
import pdfplumber as pdfplumber
import requests as requests

# National fact sheet
pdf_url = "https://www.nfhsiips.in/nfhsuser/assets/National%20Family%20Health%20Survey%20(NFHS-6)%202023-2024%20Fact%20Sheets.pdf"

def download_nfhs_pdf(url, timeout=60):
    try:
        return requests.get(url, timeout=timeout)
    except requests.exceptions.SSLError:
        intermediate = requests.get(
            "http://repository.emSign.com/certs/emSignSSLCAG1.crt",
            timeout=10,
        ).content
        intermediate_pem = subprocess.run(
            ["openssl", "x509", "-inform", "DER", "-outform", "PEM"],
            input=intermediate,
            capture_output=True,
            check=True,
        ).stdout.decode()

        with tempfile.NamedTemporaryFile(mode="w", suffix=".pem", delete=False) as bundle_file:
            with open(certifi.where()) as ca_bundle:
                bundle_file.write(ca_bundle.read())
            bundle_file.write(intermediate_pem)
            bundle_path = bundle_file.name

        try:
            return requests.get(url, verify=bundle_path, timeout=timeout)
        finally:
            os.unlink(bundle_path)

def parse_national_fact_sheet(pdf):
    """Parse India national key indicators from NFHS-6 fact sheet pages."""
    rows = []
    value_only = re.compile(r"^([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s*$")
    indicator_start = re.compile(r"^(\d+)\.")
    skip_exact = {"India - Key Indicators", "Indicators", "Urban", "Rural", "Total"}
    pending = None

    for page in pdf.pages:
        text = page.extract_text() or ""
        if "India - Key Indicators" not in text:
            continue
        for line in text.split("\n"):
            line = line.strip()
            if not line or line in skip_exact:
                continue
            if line.startswith("NFHS-") or line.startswith("(20"):
                continue

            value_match = value_only.match(line)
            if value_match and pending:
                rows.append(
                    {
                        "indicator_no": pending["no"],
                        "indicator": " ".join(pending["parts"]).strip(),
                        "urban_nfhs6": float(value_match.group(1)),
                        "rural_nfhs6": float(value_match.group(2)),
                        "total_nfhs6": float(value_match.group(3)),
                        "total_nfhs5": float(value_match.group(4)),
                    }
                )
                pending = None
                continue

            indicator_match = indicator_start.match(line)
            if indicator_match:
                indicator_no = int(indicator_match.group(1))
                rest = line[indicator_match.end():].strip()
                parts = rest.rsplit(None, 4)
                if len(parts) == 5 and all(re.match(r"^[\d.]+$", p) for p in parts[1:]):
                    rows.append(
                        {
                            "indicator_no": indicator_no,
                            "indicator": parts[0].strip(),
                            "urban_nfhs6": float(parts[1]),
                            "rural_nfhs6": float(parts[2]),
                            "total_nfhs6": float(parts[3]),
                            "total_nfhs5": float(parts[4]),
                        }
                    )
                    pending = None
                else:
                    pending = {"no": indicator_no, "parts": [rest]}
            elif pending is not None:
                pending["parts"].append(line)

    return pd.DataFrame(rows)


resp = download_nfhs_pdf(pdf_url)
resp.raise_for_status()

with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
    df_national = parse_national_fact_sheet(pdf)

df_national.head(20)

,indicator_no,indicator,urban_nfhs6,rural_nfhs6,total_nfhs6,total_nfhs5
0,1,Population below age 5 years (%),6.6,8.6,8.0,8.2
1,2,Population below age 15 years (%),22.0,27.0,25.5,26.5
2,3,Population age 60 years and above (%),12.7,13.0,12.9,11.8
3,4,Population living in households with electrici...,99.5,97.8,98.3,96.8
4,5,Population living in households with an improv...,99.1,95.4,96.5,95.9
5,6,Households using iodized salt (%),96.2,93.3,94.2,94.3
6,7,Households with any usual member covered under...,56.4,62.0,60.2,41.0
7,8,Households with any usual member having a bank...,97.3,98.6,98.2,95.7
8,9,Female population age 6 years and above who ev...,84.3,69.2,73.7,71.8
9,10,Households with any usual female members ownin...,18.2,19.1,18.8,14.0
